# Natural Gas Headline Filter using LLM

This notebook filters news headlines from `gs://codeml/gas_headlines.json` to keep only those relevant to natural gas, natural gas prices, energy, or oil.

**Data Source:**
- Input: `gs://codeml/gas_headlines.json`

**Output:**
- Filtered headlines: `gas_headlines_filtered.json`

## 1. Install Dependencies

In [ ]:
!pip install requests pandas tqdm

## 2. Import Libraries and Configure API

In [ ]:
import requests
import json
import pandas as pd
from tqdm import tqdm
from typing import Dict, Any

# API Configuration
API_ENDPOINT = "https://likely-flowing-shrimp.ngrok-free.app/v1/chat/completions"
MODEL_NAME = "hoangquan456/qwen3-nothink:8b"

print("Libraries imported successfully")

## 3. Define Relevance Classification Function

In [ ]:
def check_headline_relevance(headline: str) -> bool:
    """
    Check if a headline is relevant to natural gas, natural gas prices, energy, or oil.
    
    Args:
        headline: The news headline to check
        
    Returns:
        bool: True if relevant, False otherwise
    """
    # Construct the prompt
    system_prompt = "You are a news classification assistant. Your task is to determine if a news headline is relevant to natural gas, natural gas prices, energy, or oil. Return only a JSON object with no markdown formatting."
    
    user_prompt = f"""Is this headline relevant to natural gas, natural gas prices, energy, or oil?

Headline: {headline}

Return only a JSON object in this exact format with no other text or formatting:
{{"relevant": true}}
or
{{"relevant": false}}"""
    
    # Prepare API request
    payload = {
        "model": MODEL_NAME,
        "messages": [
            {
                "role": "system",
                "content": system_prompt
            },
            {
                "role": "user",
                "content": user_prompt
            }
        ]
    }
    
    try:
        # Make API request
        response = requests.post(
            API_ENDPOINT,
            headers={"Content-Type": "application/json"},
            json=payload,
            timeout=30
        )
        
        response.raise_for_status()
        
        # Parse response
        response_data = response.json()
        assistant_message = response_data['choices'][0]['message']['content']
        
        # Clean up response (remove potential markdown formatting)
        assistant_message = assistant_message.strip()
        if assistant_message.startswith('```json'):
            assistant_message = assistant_message[7:]
        if assistant_message.startswith('```'):
            assistant_message = assistant_message[3:]
        if assistant_message.endswith('```'):
            assistant_message = assistant_message[:-3]
        assistant_message = assistant_message.strip()
        
        # Parse JSON response
        result = json.loads(assistant_message)
        return result.get('relevant', False)
        
    except Exception as e:
        print(f"Error processing headline: {headline[:50]}... - {e}")
        # Default to False if there's an error
        return False


print("Relevance classification function defined")

## 4. Test the Function with Sample Headlines

In [ ]:
# Test with sample headlines
test_headlines = [
    "Natural Gas Prices Surge Amid Cold Weather Forecast",
    "Celebrity Wedding Announcement Shocks Fans",
    "New Oil Pipeline Project Approved by Regulatory Board",
    "Local Restaurant Wins Best Pizza Award"
]

print("Testing relevance classification:\n")
for headline in test_headlines:
    is_relevant = check_headline_relevance(headline)
    print(f"Headline: {headline}")
    print(f"Relevant: {is_relevant}")
    print("-" * 80)

## 5. Load Gas Headlines Data

In [ ]:
# Load the headlines from JSON file
headlines_df = pd.read_json('gs://codeml/gas_headlines.json')

print(f"Loaded {len(headlines_df)} headlines")
print(f"\nColumns: {headlines_df.columns.tolist()}")
print(f"\nFirst 5 headlines:")
display(headlines_df.head())

## 6. Filter Headlines Using LLM

In [ ]:
# Process each headline and check relevance
filtered_headlines = []
relevance_stats = {'relevant': 0, 'not_relevant': 0}

print("Processing headlines...\n")

for index, row in tqdm(headlines_df.iterrows(), total=len(headlines_df), desc="Filtering headlines"):
    headline = row['headline']
    
    # Check if headline is relevant
    is_relevant = check_headline_relevance(headline)
    
    if is_relevant:
        # Keep the entire row if relevant
        filtered_headlines.append(row.to_dict())
        relevance_stats['relevant'] += 1
    else:
        relevance_stats['not_relevant'] += 1

# Create filtered DataFrame
filtered_df = pd.DataFrame(filtered_headlines)

# Display statistics
print("\n" + "="*60)
print("FILTERING RESULTS")
print("="*60)
print(f"Total headlines processed: {len(headlines_df)}")
print(f"Relevant headlines: {relevance_stats['relevant']}")
print(f"Not relevant headlines: {relevance_stats['not_relevant']}")
print(f"Retention rate: {(relevance_stats['relevant'] / len(headlines_df) * 100):.2f}%")
print("="*60)

print(f"\nFiltered headlines sample:")
display(filtered_df.head(10))

## 7. Save Filtered Headlines to JSON

In [ ]:
# Save to JSON file (same format as input)
output_file = 'gas_headlines_filtered.json'

# Convert DataFrame to list of dictionaries and save
filtered_data = filtered_df.to_dict('records')

with open(output_file, 'w', encoding='utf-8') as f:
    json.dump(filtered_data, f, ensure_ascii=False, indent=2)

print(f"Filtered headlines saved to: {output_file}")
print(f"Total entries saved: {len(filtered_data)}")

# Verify the output file
print(f"\nVerifying output file...")
with open(output_file, 'r', encoding='utf-8') as f:
    verification_data = json.load(f)
    print(f"Verification: Successfully loaded {len(verification_data)} entries from output file")
    print(f"\nFirst entry in output file:")
    print(json.dumps(verification_data[0], indent=2))

## Summary

### Process Overview
1. **Data Loading**: Loaded news headlines from `gs://codeml/gas_headlines.json`
2. **LLM Classification**: Used Qwen3-nothink model to classify each headline's relevance
3. **Filtering**: Kept only headlines relevant to natural gas, natural gas prices, energy, or oil
4. **Output**: Saved filtered headlines to `gas_headlines_filtered.json`

### Key Features
- Uses LLM endpoint with structured JSON output
- Cleans response to handle potential markdown formatting
- Preserves original data structure (headline, date, summary)
- Provides detailed filtering statistics
- Error handling for API failures

### Next Steps
- Use `gas_headlines_filtered.json` as input for sentiment analysis
- Potentially fine-tune filtering criteria based on results
- Consider batch processing for large datasets
- Implement caching to avoid re-processing headlines

## 8. Run Complete Pipeline (All Steps Combined)

In [ ]:
import requests
import json
import pandas as pd
from tqdm import tqdm
from typing import Dict, Any, List, Tuple
from concurrent.futures import ThreadPoolExecutor, as_completed
import time

print("="*80)
print("NATURAL GAS HEADLINE FILTERING PIPELINE (BATCH PROCESSING)")
print("="*80)

# === CONFIGURATION ===
# API_ENDPOINT = "https://likely-flowing-shrimp.ngrok-free.app/v1/chat/completions"
API_ENDPOINT = "https://khh6fxbehjar.share.zrok.io/v1/chat/completions"
# MODEL_NAME = "hoangquan456/qwen3-nothink:8b"
MODEL_NAME = "hoangquan456/qwen3-nothink:1.7b"
INPUT_FILE = "gs://codeml/gas_headlines.json"
# INPUT_FILE = "gs://codeml/ngi.json"
# INPUT_FILE = "gs://codeml/eia.json"
OUTPUT_FILE = "gas_headlines_filtered.json"
MAX_WORKERS = 6  # Number of concurrent batch requests
BATCH_SIZE = 12  # Number of headlines per batch
REQUEST_TIMEOUT = 5  # Timeout in seconds for API requests
MAX_RETRIES = 3  # Maximum number of retry attempts
INITIAL_RETRY_DELAY = 1  # Initial delay between retries in seconds

print(f"\nConfiguration:")
print(f"  Model: {MODEL_NAME}")
print(f"  Input: {INPUT_FILE}")
print(f"  Output: {OUTPUT_FILE}")
print(f"  Batch size: {BATCH_SIZE}")
print(f"  Concurrent workers: {MAX_WORKERS}")
print(f"  Request timeout: {REQUEST_TIMEOUT}s")
print(f"  Max retries: {MAX_RETRIES}")
print(f"  Initial retry delay: {INITIAL_RETRY_DELAY}s")


# === DEFINE BATCH CLASSIFICATION FUNCTION ===
def check_headlines_batch_relevance(headlines_batch: List[Tuple[int, str]]) -> List[int]:
    """
    Check if headlines are relevant to natural gas industry with retry logic.
    
    Args:
        headlines_batch: List of tuples (index, headline)
        
    Returns:
        list: Indices of relevant headlines (empty list if all retries fail)
    """
    system_prompt = """You are a news classification assistant specialized in the natural gas industry. 
Your task is to identify headlines with STRICT relevance to natural gas, natural gas prices, or the natural gas industry.

STRICT RELEVANCE CRITERIA:
- Must DIRECTLY mention natural gas, LNG (liquefied natural gas), or natural gas industry companies
- Must discuss current or future events/forecasts (NOT past/historical events)
- Slight implications or hints are NOT sufficient
- General energy topics without specific natural gas mention are NOT relevant
- Oil-only topics are NOT relevant unless they mention natural gas impact

Return only a JSON object with no markdown formatting."""
    
    # Build the headline list
    headlines_text = ""
    for idx, headline in headlines_batch:
        headlines_text += f"{idx}: {headline}\n"
    
    user_prompt = f"""Review these {len(headlines_batch)} headlines and return ONLY the indices of headlines that are STRICTLY relevant to natural gas.

Headlines:
{headlines_text}

STRICT Requirements:
1. Must DIRECTLY mention "natural gas", "LNG", or natural gas industry companies
2. Must discuss CURRENT or FUTURE events (no historical/past events)
3. Must be specifically about natural gas (not just general energy/oil)
4. Vague implications are NOT sufficient

Return ONLY a JSON object in this exact format with no other text:
{{"relevant_indices": [1, 3, 7]}}

If NO headlines are relevant, return:
{{"relevant_indices": []}}"""
    
    payload = {
        "model": MODEL_NAME,
        "messages": [
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt}
        ]
    }
    
    # Retry loop with exponential backoff
    for attempt in range(MAX_RETRIES):
        try:
            response = requests.post(
                API_ENDPOINT,
                headers={"Content-Type": "application/json"},
                json=payload,
                timeout=REQUEST_TIMEOUT
            )
            response.raise_for_status()
            
            # Parse and clean response
            assistant_message = response.json()['choices'][0]['message']['content'].strip()
            
            # Remove markdown formatting if present
            if assistant_message.startswith('```json'):
                assistant_message = assistant_message[7:]
            if assistant_message.startswith('```'):
                assistant_message = assistant_message[3:]
            if assistant_message.endswith('```'):
                assistant_message = assistant_message[:-3]
            assistant_message = assistant_message.strip()
            
            # Parse JSON
            result = json.loads(assistant_message)
            return result.get('relevant_indices', [])
            
        except requests.exceptions.Timeout as e:
            if attempt < MAX_RETRIES - 1:
                delay = INITIAL_RETRY_DELAY * (2 ** attempt)
                time.sleep(delay)
                continue
            else:
                print(f"  Timeout after {MAX_RETRIES} attempts (batch size: {len(headlines_batch)})")
                return []
                
        except requests.exceptions.HTTPError as e:
            if attempt < MAX_RETRIES - 1:
                if e.response.status_code in [504, 502, 503]:
                    delay = INITIAL_RETRY_DELAY * (2 ** attempt)
                    time.sleep(delay)
                    continue
                else:
                    print(f"  HTTP error {e.response.status_code}: {str(e)[:80]}")
                    return []
            else:
                print(f"  HTTP error after {MAX_RETRIES} attempts: {e.response.status_code}")
                return []
                
        except json.JSONDecodeError as e:
            if attempt < MAX_RETRIES - 1:
                delay = INITIAL_RETRY_DELAY * (2 ** attempt)
                time.sleep(delay)
                continue
            else:
                print(f"  JSON parse error after {MAX_RETRIES} attempts: {str(e)[:80]}")
                return []
                
        except requests.exceptions.RequestException as e:
            if attempt < MAX_RETRIES - 1:
                delay = INITIAL_RETRY_DELAY * (2 ** attempt)
                time.sleep(delay)
                continue
            else:
                print(f"  Network error after {MAX_RETRIES} attempts: {str(e)[:80]}")
                return []
                
        except Exception as e:
            print(f"  Unexpected error: {str(e)[:100]}")
            return []
    
    return []


def process_batch(batch_data: List[Tuple[int, Any]]) -> List[Dict]:
    """Process a batch of rows and return results with relevance status."""
    # batch_data is a list of (index, row) tuples
    headlines_batch = []
    for idx, row in batch_data:
        # Concatenate headline with summary if it exists
        text = row['headline']
        if 'summary' in row and pd.notna(row['summary']):
            text = f"{row['headline']} {row['summary']}"
        headlines_batch.append((idx, text))

    relevant_indices = check_headlines_batch_relevance(headlines_batch)
    
    # Convert to set for O(1) lookup
    relevant_set = set(relevant_indices)
    
    results = []
    for idx, row in batch_data:
        results.append({
            'index': idx,
            'row': row.to_dict(),
            'is_relevant': idx in relevant_set
        })
    
    return results


# === LOAD DATA ===
print(f"\n{'='*80}")
print("STEP 1: Loading Data")
print(f"{'='*80}")

headlines_df = pd.read_json(INPUT_FILE)
print(f"✓ Loaded {len(headlines_df)} headlines")
print(f"  Columns: {headlines_df.columns.tolist()}")


# === CREATE BATCHES ===
print(f"\n{'='*80}")
print("STEP 2: Creating Batches")
print(f"{'='*80}")

batches = []
current_batch = []

for index, row in headlines_df.iterrows():
    current_batch.append((index, row))
    if len(current_batch) == BATCH_SIZE:
        batches.append(current_batch)
        current_batch = []

# Add remaining items as last batch
if current_batch:
    batches.append(current_batch)

print(f"✓ Created {len(batches)} batches")


# === FILTER HEADLINES WITH BATCH CONCURRENCY ===
print(f"\n{'='*80}")
print("STEP 3: Filtering Headlines with LLM (Batch Processing)")
print(f"{'='*80}")

filtered_headlines = []
relevance_stats = {'relevant': 0, 'not_relevant': 0, 'failed': 0}

# Use ThreadPoolExecutor for concurrent batch processing
with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
    # Submit all batch tasks
    futures = {
        executor.submit(process_batch, batch): batch 
        for batch in batches
    }
    
    # Process completed futures with progress bar
    for future in tqdm(as_completed(futures), total=len(futures), desc="Processing batches"):
        try:
            results = future.result()
            
            for result in results:
                if result['is_relevant']:
                    filtered_headlines.append(result['row'])
                    relevance_stats['relevant'] += 1
                else:
                    relevance_stats['not_relevant'] += 1
                    
        except Exception as e:
            print(f"  Critical batch processing error: {str(e)[:100]}")
            # Count all items in failed batch as failed
            batch = futures[future]
            relevance_stats['failed'] += len(batch)

# Create filtered DataFrame
filtered_df = pd.DataFrame(filtered_headlines)


# === DISPLAY RESULTS ===
print(f"\n{'='*80}")
print("FILTERING RESULTS")
print(f"{'='*80}")
print(f"Total headlines processed: {len(headlines_df)}")
print(f"Relevant headlines: {relevance_stats['relevant']}")
print(f"Not relevant headlines: {relevance_stats['not_relevant']}")
print(f"Failed to process: {relevance_stats['failed']}")
processed_count = relevance_stats['relevant'] + relevance_stats['not_relevant']
if processed_count > 0:
    print(f"Retention rate: {(relevance_stats['relevant'] / processed_count * 100):.2f}%")
if relevance_stats['failed'] > 0:
    print(f"Failed rate: {(relevance_stats['failed'] / len(headlines_df) * 100):.2f}%")


# === SAVE OUTPUT ===
print(f"\n{'='*80}")
print("STEP 4: Saving Filtered Data")
print(f"{'='*80}")

filtered_df_copy = filtered_df.copy()
for col in filtered_df_copy.select_dtypes(include=['datetime64']).columns:
    filtered_df_copy[col] = filtered_df_copy[col].dt.strftime('%Y-%m-%d')

filtered_data = filtered_df_copy.to_dict('records')

with open(OUTPUT_FILE, 'w', encoding='utf-8') as f:
    json.dump(filtered_data, f, ensure_ascii=False, indent=2)

print(f"✓ Filtered headlines saved to: {OUTPUT_FILE}")
print(f"✓ Total entries saved: {len(filtered_data)}")


# === VERIFY OUTPUT ===
print(f"\n{'='*80}")
print("VERIFICATION")
print(f"{'='*80}")

with open(OUTPUT_FILE, 'r', encoding='utf-8') as f:
    verification_data = json.load(f)
    print(f"✓ Successfully verified {len(verification_data)} entries in output file")

print(f"\n{'='*80}")
print("PIPELINE COMPLETE")
print(f"{'='*80}")
print(f"\nFiltered headlines preview:")
print(filtered_df.head(10))